# Downloading sequencing data, quantification and processing data for QC

## Table of Contents:

* [0. Dependencies](#Dependencies)
* [1. Preparing the environment](#Preparing-the-environment)
* [2. Downloading sequencing data from ENA](#Downloading-sequencing-data-from-ENA)
* [3. Preparing library annotations](#Preparing-library-annotations)
* [4. Quantification](#Quantification)
* [5. Removing user-defined guides](#Removing-user-defined-guides)
* [6. Normalised count matrix](#Normalised-count-matrix)
* [7. Filtering of guides with low counts in control samples](#Filtering-of-guides-with-low-counts-in-control-samples)
* [8. Normalised log fold changes](#Normalised-log-fold-changes)

***

## Dependencies

Please see [INSTALL_README.md](../../INSTALL_README.md) for the installation of software dependencies. This Notebook assumes that [R](https://cran.r-project.org/) and library dependencies have been installed and that the `RScript` command is available with access to the relevant data.

***

## Preparing the environment

Several paths are used as input to more than one script. To simplify the commands and improve usability, commonly used paths are stored as environment variables. Those variables are then used in the script arguments to shorten input and output data paths. 

This Notebook assumes that you have the following directory structure and files in place before running the commands: 

* **Top level directory** (`REPO_PATH`)
    * **DATA**
        * **sequencing** (`SEQ_PATH`)
            * *\*.cram*
        * **pyCROQUET** (`PYCROQUET_OUTPUT_PATH`)
        * **preprocessing** (`TSV_PATH`)
        * **RDS** (`RDS_PATH`)
    * **LOGS**
        * pyCROQUET (`PYCROQUET_LOG_PATH`)
    * **METADATA**
        * *guide_ids_to_remove.txt*
        * *safe_targeting_guides.txt*
        * *sample_annotations.tsv*
        * *sampled_to_remove.txt*
        * **libraries**
            * *paralog_library.tsv*

The scripts require several files to be present:

* `METADATA/sample_annotations.tsv` - sample metadata.
* `METADATA/safe_targeting_guides.txt` - sgRNA identifiers of safe-targeting guides (one per line) which are paired with a gene-targeting guide in single constructs and another safe-targeting guide in control constructs
* `METADATA/guide_ids_to_remove.txt` - paired construct identifiers to remove in guide filtering step
* `METADATA/samples_to_remove.txt` - sample identifiers to remove (based on QC) before scaling 
* `METADATA/libraries/paralog_library.tsv` - paralog library anotations (one row per paired construct)

In [ ]:
# Set the top level path for the repository
export REPO_PATH=$(dirname `pwd`)

# Show the repository path
echo "Repository path: ${REPO_PATH}"

# Check the repository path exists (don't need to check subdirectories)
if [ ! -d "${REPO_PATH}" ]; then
  echo "Top level directory path does not exist: ${REPO_PATH}"
fi

Once the top level directory has been set (this will likely be the path to your clone of the repository), we then set several other resusable paths as environment variables and create their directories if they don't exist. It should not be assumed this exist when you clone the repository as they may be present in the `.gitignore` files (e.g. output logs or large data files).

In [ ]:
# Set environment variables for reusable paths
export LOG_PATH="${REPO_PATH}/LOGS"
export TSV_PATH="${REPO_PATH}/DATA/preprocessing"
export RDS_PATH="${REPO_PATH}/DATA/RDS/preprocessing"

# Show the paths (debug)
echo "Log path: ${LOG_PATH}"
echo "TSV output directory: ${TSV_PATH}"
echo "RDS output directory: ${RDS_PATH}"

# Create the directories if they don't exist 
mkdir -p "${LOG_PATH}"
mkdir -p "${TSV_PATH}"
mkdir -p "${RDS_PATH}"

## Create directories required by pyCROQUET

# CRAM path
export SEQ_PATH="${REPO_PATH}/DATA/sequencing"
echo "Sequencing data path: ${SEQ_PATH}"
mkdir -p "${SEQ_PATH}"

# pyCROQUET output path
export PYCROQUET_OUTPUT_PATH="${REPO_PATH}/DATA/pyCROQUET"
echo "pyCROQUET output path: ${PYCROQUET_OUTPUT_PATH}"
mkdir -p "${PYCROQUET_OUTPUT_PATH}"

# pyCROQUET log path
export PYCROQUET_LOG_PATH="${LOG_PATH}/pyCROQUET"
echo "pyCROQUET log path: ${PYCROQUET_LOG_PATH}"
mkdir -p "${PYCROQUET_LOG_PATH}"

*** 

## Downloading sequencing data from ENA

It is possible to download FASTQ or SRA files from the [ENA](https://www.ebi.ac.uk/ena/browser/home), however, at the time of analysis, the processing of the public files in project [PRJEB60853](https://www.ebi.ac.uk/ena/browser/view/PRJEB60853) was incomplete. Instead of using a pipeline (e.g. []()) which can automatically pull the processed data, we have opted for pulling the submitted CRAMs from the ENA FTP. 

Selected ENA sample metadata was downloaded and saved in `METADATA/ena_sample_metadata.tsv`. We used the `submitted_ftp` path to download the submitted CRAMs for each sample and store them in `DATA/sequencing`. There are 516 CRAMs across 96 samples.


In [ ]:
while read ftp_path
do
    modified_ftp_path=${ftp_path/\#/"%23"}
    base_filename=$(basename -- $ftp_path)
    echo "Downloading ${base_filename}"
    if [ -e "${SEQ_PATH}/${base_filename}" ]
    then
        echo "File exists: ${SEQ_PATH}/${base_filename}"
    else
        wget -q --directory-prefix="${SEQ_PATH}" "$modified_ftp_path"
        if [ -e "${SEQ_PATH}/${base_filename}" ]
        then
            echo "File written to: ${SEQ_PATH}/${base_filename}"
        else
            echo "File not written to: ${SEQ_PATH}/${base_filename}"
        fi
    fi
done < <(awk -F '\t' 'NR > 1 {print $19}' ${REPO_PATH}/METADATA/ena_sample_metadata.tsv )

***

## Preparing library annotations

Library annotations can be found in `METADATA/libraries/paralog_library.tsv` which contains information about the combinatorial construct. The sources of the guides are: `TO` - Toronto, `BR` - Brunello, `LS`- Lander Sabatini, `CO`- Croatan and `BG` - Broad GPP server. To allow for comparison with existing libraries, the `sorted_gene_pair` field has been provided where the two targeted genes are sorted alphabetically. The position of those genes and guides can be determined form the `sgrna_ids`, `targetA`, `targetB`, `sgrnaA`, `sgrnaB`, ` sgrna_group` and `guide_type` fields.

| Column name | Description |
| --- | --- |
| id | unique identifier for the guide pair - includes gene pair as guides may be used in multiple gene pairs (note: this field is arbitrary) |
| sgrna_ids | individual guide identifiers in the format `positionA\|positionB` |
| sgrna_seqs | individual guide sequences in the format `positionA\|positionB` |
| sorted_gene_pair | alphabetically sorted gene symbols for genes targeted by guide pair in the format `geneA\|geneB` (note: these are sorted alphabetically and do not denote the position in the construct) |
| targetA | gene symbol for target of guide in position A in the construct |
| targetB | gene symbol for target of guide in position B in the construct |
| sgrnaA | guide identifier of guide in position A in the construct |
| sgrnaB | guide identifier of guide in position B in the construct |
| sgrna_source | source of the guides in the format `positionA\|positionB` (see below for source descriptions) |
| sgrna_group | source of the targeted pairs in the format `positionA\|positionB` (see below for source descriptions) |
| guide_type | whether the pair is dual (`gene\|gene`), single (`gene\|safe_targeting` or `safe_targeting\|gene`) or safe (`safe_targeting\|safe_targeting`) |
| guide_orientation | position of targets and guides in a pair (e.g. `geneA\|geneB`, `geneB\|geneA`, `safe_targeting\|geneB`, `safe_targeting\|geneA`) |
| singles_target_gene | gene symbol for target of a single guide (e.g. where `guide_type` is `gene\|safe_targeting` or `safe_targeting\|gene`) |

In this step, the library annotations (`METADATA/libraries/paralog_library.tsv`) are formatted for use with [pyCROQUET](https://github.com/cancerit/pycroquet) using the guidance in the [pyCROQUET Wiki](https://github.com/cancerit/pycroquet/wiki/Guide-library-format) (`METADATA/paralog_library.pycroquet.tsv`). The pyCROQUET library (`METADATA/paralog_library.pycroquet.tsv`) is a non-redundant version of `METADATA/libraries/paralog_library.tsv` which allows for the quantification of a smaller number or unique guide pairs identified by the `sgrna_ids`, `sgrnaA` and `sgrnaB`. The counts are expanded using the redundant library (`METADATA/libraries/paralog_library.tsv`) when collating the count matrix.

We cannot use this step to deterime the library for the singles analysis with [MAGeCK](https://sourceforge.net/p/mageck) and [BAGEL2](https://github.com/hart-lab/bagel) and they use processed (filtered, normalised and scaled) data. However, this step does generate the *dual guide matrix* is for the Bassik and Hart analyses (`METADATA/libraries/dual_guide_matrix.tsv`) linking the identifiers of the single guides (`g1` and `g2`) to their paired construct identifier for each combinatorial guide pair (`g1g2`).

In [ ]:
echo "Preparing libraries for analysis..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/00_prepare_libraries.R \
	-d ${REPO_PATH} \
	-m ${REPO_PATH}/METADATA/sample_annotations.tsv \
	-a ${REPO_PATH}/METADATA/libraries/paralog_library.tsv

***

## Quantification

The paralog library used for quantification is redundant - the same individual guide pair (e.g. safe-targeting guide + gene-targeting guide) may be used for  multiple guide pairs. 

| unique_id | sorted_gene_pair | sgrnaA | sgrnaB |
| --- | --- | --- | --- |
| F5\|chr16:29874121-29874140_CDIPT_+\|TSTD1\|CDIPT | CDIPT\|**TSTD1** | F5 | chr16:29874121-29874140_CDIPT_+ |
| F5\|chr16:29874121-29874140_CDIPT_+\|PCDH1\|CDIPT | CDIPT\|**PCDH1** | F5 | chr16:29874121-29874140_CDIPT_+ |
| F5\|chr16:29874121-29874140_CDIPT_+\|IRF6\|CDIPT |  CDIPT\|**IRF6** | F5 | chr16:29874121-29874140_CDIPT_+ |

While we could quantify against a non-redundant library and expand counts at a later stage, the redundant library is sufficiently small that quantifying is not computationally expensive.

[pyCROQUET](https://github.com/cancerit/pycroquet) version 1.5.1 was used to quantify samples using the redundant library (`paralog_library.pycroquet.tsv`) which was processed/formatted in the previous step. Each sample was sequenced across multiple seuqencing runs with quantification performed for each CRAM with counts and statistics subsequently merged.

As the quantification can take a while to run, these were submitted as a job array via LSF on the Sanger compute farm (`SCRIPTS/quantification/pycroquet_lsf_jobscript.sh`). *Note: there are hard coded paths in this file which will need to be updated if you are intending to use it*.

An example command to run pyCROQUET outside LSF (per CRAM) would be:

```
pycroquet dual-guide -g METADATA/libraries/paralog_library.pycroquet.tsv -q "${SEQ_PATH}/33677_1#1.cram" -o "${PYCROQUET_OUTPUT_PATH}/33677_1#1" --chunks 50000 -b exact
```

where:
* `${guides}` is the pyCROQUET-formatted library (METADATA/paralog_library.pycroquet.tsv)
* `${cram_directory}/${query_file_name}` is the CRAM file being quantified
* `"${output_directory}/${query_file_label}` is the output path and file prefix (derived from the file name)
* `--chunks 50000` see [here](https://github.com/cancerit/pycroquet#chunks) for an explanation of how to use the `--chunks` option to modify/optimise CPU usage

To run with LSF (assumes pyCROQUET is available as a module):

```
cd ${REPO_PATH}
bsub < ${REPO_PATH}/SCRIPTS/quantification/pycroquet_lsf_jobscript.sh
```

For each CRAM the following output files are generated:

* `[run]_[lane]#[tag].counts.tsv.gz` - frequency of each paired construct (*excluded from repository due to size*)
* `[run]_[lane]#[tag].cram` and `[run]_[lane]#[tag].cram.crai` - alignment of reads to guides (*excluded from repository due to size*)
* `[run]_[lane]#[tag].query_class.tsv.gz` - classification of each read pair (*excluded from repository due to size*)
* `[run]_[lane]#[tag].stats.json` - summary statistics

Individual counts need to be collated into sample counts where the sample count is the sum of all individual counts (per-guide pair) for that sample. As quantification was against the non-redundant library of unique guide pairs (`METADATA/paralog_library.pycroquet.tsv`), the counts are expanded for analysis using the non-redudant (gene-pair-dependent) version of the library (`METADATA/paralog_library.tsv`). The raw count matrix has been written to: `DATA/preprocessing/count_matrix.tsv`.

In [ ]:
echo "Converting raw lane counts into raw sample count matrix..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/01_merge_counts_by_sample.R \
	-d ${REPO_PATH} \
	-m ${REPO_PATH}/METADATA/sample_annotations.tsv \
	-a ${REPO_PATH}/METADATA/libraries/paralog_library.tsv

*** 

## Removing user-defined guides

The following script allows the user to provide a file with a list of paired guide construct identifiers for removal (`METADATA/guides_ids_to_remove.txt`). Here, we remove 60 guides.

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.tsv | wc -l

In [ ]:
tail -n +2 ${REPO_PATH}/METADATA/libraries/paralog_library.tsv | wc -l

In [ ]:
echo "Removing user-defined guides from count matrix..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/02_remove_user_defined_guides.R \
	-d ${REPO_PATH} \
	-g ${REPO_PATH}/METADATA/guides_ids_to_remove.txt \
	-c ${REPO_PATH}/DATA/preprocessing/count_matrix.tsv

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.guides_removed.tsv | wc -l

***

## Normalised count matrix

A pseudocount of 5 is added to the raw, per-sample counts which are then total normalised using a scaling factor of 10 million reads.

In [ ]:
echo "Normalising counts using BAGEL method..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/03_bagel_normalisation.R \
	-d ${REPO_PATH} \
	-c ${REPO_PATH}/DATA/preprocessing/count_matrix.guides_removed.tsv \
    --annotations 13

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.norm.tsv | wc -l

***

## Filtering of guides with low counts in control samples

Before calculating normalised fold changes, guides with low counts (< 20 reads per 10 million)in control samples are removed. First, the columns representing control samples (defined by the user) are extracted from the normalised count matrix. The `rowMean` is calculated as the mean of the normalised control counts for each paired guide contstruct. Guides are removed when the normalised mean control count is less than the user-defined minimum value. Here, we remove 51 paired guides with mean normalised control sample counts less than 20 (reads per 10 million).

In [ ]:
echo "Filter guides with low control counts..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/04_filter_low_count_guides.R \
	-d ${REPO_PATH} \
	-c ${REPO_PATH}/DATA/preprocessing/count_matrix.norm.tsv \
	-m ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --filter 20 \
    --samples 'MeWo control R1,MeWo control R2,MeWo control R3,Capan-1 control R1,Capan-1 control R2,Capan-1 control R3,A549 control R1,A549 control R2,A549 control R3'

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.norm.tsv | wc -l

In [ ]:
tail -n +1 ${REPO_PATH}/DATA/preprocessing/filtered_guides.txt | wc -l

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.tsv | wc -l

*** 

## Normalised log fold changes

Normalised counts are converted into normalised log fold changes. First, the rowMeans of the user-defined control samples are re-calculated and added to the normalised count matrix `DATA/preprocessing/count_matrix.norm.filt.mean_control.tsv`. Then a pseudocount of 0.5 is added to the counts before calculating the log fold change for each paired guide contstuct as `log2(normalised sample count /control mean)`. 

The normalised log fold change matrix is written to: `DATA/preprocessing/lfc_matrix.unscaled.tsv`.

In [ ]:
echo "Convert count matrix to LFC matrix..."
Rscript ${REPO_PATH}/SCRIPTS/preprocessing/05_convert_counts_to_lfcs.R \
	-d ${REPO_PATH} \
	-c ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.tsv \
	-m ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --annotations 13 \
    --samples 'MeWo control R1,MeWo control R2,MeWo control R3,Capan-1 control R1,Capan-1 control R2,Capan-1 control R3,A549 control R1,A549 control R2,A549 control R3' 

In [ ]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/lfc_matrix.unscaled.tsv | wc -l